# Value exploration

Establishes what the correct answers actually are, by reading the document,
before any model is asked to extract them.

This has to come first. Without known answers there is no way to tell a correct
extraction from a plausible one - and as the cells below show, this document
offers many plausible wrong answers for every field.

The values confirmed here are written to `evaluation/expected.yaml` and used to
score every later run.

In [1]:
import re
import sys
from pathlib import Path

import pypdf

sys.path.insert(0, str(Path.cwd().parent))

from src.config import load_config

config = load_config(Path.cwd().parent / "config.yml")
reader = pypdf.PdfReader(str(Path.cwd().parent / config.pdf_path))


def find(term, context=95):
    """Print every appearance of a term across the whole document."""
    for page_no, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        for match in re.finditer(term, text):
            excerpt = " ".join(text[match.start() : match.start() + context].split())
            print(f"  p{page_no:2d}: {excerpt}")


print(f"{len(reader.pages)} pages")

37 pages


## The document reports two fiscal years at once

This is the root of nearly every ambiguity below. Section 1 revises the year
just ending; section 2 estimates the year being budgeted for. Both are present,
both are labelled, and a figure means nothing without knowing which section it
came from.

In [2]:
for page_no in (5, 13):
    text = reader.pages[page_no - 1].extract_text() or ""
    heading = " ".join(text[:220].split())
    print(f"p{page_no}: {heading}\n")

p5: MINISTRY OF FINANCE 5 01 Update on Financial Year 2023 1.1 Expected Overall Fiscal Position for FY2023 The basic deficit is revised to $5.4 billion (0.8% of GDP). After factoring in Top- ups to Endowment and

p13: MINISTRY OF FINANCE 13 02 Outlook for Financial Year 2024 2.1 Budget for FY2024 A basic deficit of $6.1 billion (0.8% of GDP) is estimated for FY2024. After factoring in Top-ups to Endowment and Trust Funds



## Field 1-2: Corporate Income Tax, and its year-on-year change

Requirements cite page 5.

In [3]:
find("Corporate Income Tax")

  p 5: Corporate Income Tax, Other Taxes, Vehicle Quota Premiums, Personal Income Tax, Assets Taxes,
  p 5: Corporate Income Tax collections are revised to $ 28.4 billion, which is $4.1 b


  p 8: Corporate Income Tax 23.07 24.26 28.38 23.0 17.0 Personal Income Tax 15.52 16.84 17.53 12.9 4.
  p 9: Corporate Income Tax 27.2% Personal Income Tax 16.8% Goods and Services Tax 15.7% Other Taxes 8


  p16: Corporate Income Tax 28.38 28.03 (0.35) (1.2) Personal Income Tax 17.53 18.07 0.55 3.1 Withho


  p26: Corporate Income Tax 16,032 16,732 16,112 18,196 23,072 28,380 28,029 Personal Income Tax 11,7
  p27: Corporate Income Tax 3.1% 3.3% 3.3% 3.0% 3.4% 4.1% 3.9% Personal Income Tax 2.3% 2.4% 2.6% 2.3


  p37: Corporate Income Tax, Personal Income Tax, and Goods and Services Tax. Operating Expenditu


Eight appearances across seven pages, and nearly every one carries a different
number:

| Page | Value | What it is |
|---|---|---|
| 5 | **28.4** | Revised FY2023, stated in prose - **the cited page** |
| 8 | 28.38 | the same figure at table precision |
| 8 | 23.07 / 24.26 | FY2022 actual / FY2023 estimated |
| 9 | 27.2% | share of Operating Revenue, not an amount |
| 16 | 28.03 | Estimated **FY2024** - a different year |
| 26 | 28,380 / 28,029 | the same figures in $million |
| 27 | 3.9% | share of GDP |

Every one is a plausible answer to "the amount of Corporate Income Tax". They
differ by year, by unit, and by whether they are an amount or a proportion.
Nothing in the number itself distinguishes them.

**Decision: 28.4 from page 5.** The requirements cite page 5, and page 5 states
the figure in prose. Note this is less precise than page 8's 28.38 - the
citation is taken as authoritative over precision.

The year-on-year change appears in the same sentence.

In [4]:
text = reader.pages[4].extract_text() or ""
match = re.search(r"Corporate Income Tax collections[^.]+\.", text)
print(" ".join(match.group(0).split()))

Corporate Income Tax collections are revised to $ 28.


**Decision: 17.0%.** Stated in the text, not calculated. Worth noting what it is
a change *from*: the Estimated FY2023 figure, not the previous financial year.
A model asked for "year-on-year change" might reasonably compute something else,
which is why the prompt says to take the stated percentage.

## Field 3: Total top-ups

Requirements cite page 20. This is the field the first prompt got wrong.

In [5]:
find("Top-ups to Endowment")

  p 7: Top-ups to Endowment and Trust Funds, is $27.2 billion, which is $7.6 billion (38.7%) higher t
  p 8: Top-ups to Endowment and Trust Funds 2.69 2.76 2.85 COL Special Payment 1.02 1.33 1.55 CD
  p 8: Top-ups to Endowment and Trust Funds 6.25 16.82 24.32 Majulah Package Fund - - 7.50 Natio
  p 8: Top-ups to Endowment Funds4 - 2.30 2.30 Other Funds5 1.05 5.72 5.72 Add: 22,376, 23,480,5
  p 8: Top-ups to Endowment and Trust Funds. 3 Includes GST Voucher Special Payment, Top-ups to Edus
  p11: Top-ups to Endowment and Trust Funds Cost-of-Living Special Payment 1,549 CDC Vouchers 635 O
  p11: Top-ups to Endowment and Trust Funds Majulah Package Fund 7,500 National Productivity Fund 4,
  p11: Top-ups to Endowment and Trust Funds. 2Includes GST Voucher Special Payment, Top-ups to Edusav


  p13: Top-ups to Endowment and Trust Funds of $20.4 billion, NIRC of $23.5 billion, Capitalisation o


  p16: Top-ups to Endowment and Trust Funds 2.85 2.94 CDC Vouchers 0.64 0.85 COL Special Payment
  p16: Top-ups to Endowment and Trust Funds 24.32 20.35 Majulah Package Fund 7.50 - GST Voucher
  p16: Top-ups to Endowment and Trust Funds. 3 Other Transfers in FY2024 include MediSave Bonus, U -S
  p18: Top-ups to Endowment and Trust Funds ($20.4 billion) In Budget 2024, the Government will top
  p20: Top-ups to Endowment and Trust Funds in FY2024 Estimated FY2024 ($ million) Goods and Ser


  p24: Top-ups to Endowment and Trust Funds. Table 3.1a: Overall Fiscal Po
  p24: Top-ups to Endowment and Trust Funds 1,689 1,561 33,502 6,828 2,691 2,849 2,944 Basic Surplus
  p24: Top-ups to Endowment and Trust Funds 7,300 13,568 17,320 - 6,250 24,320 20,352 Net Investment
  p25: Top-ups to Endowment and Trust Funds. Table 3.1b: Overall Fiscal Position
  p25: Top-ups to Endowment and Trust Funds 0.3% 0.3% 6.8% 1.1% 0.4% 0.4% 0.4% Basic Surplus / Defici
  p25: Top-ups to Endowment and Trust Funds 1.4% 2.6% 3.5% 0.0% 0.9% 3.5% 2.8% Net Investment Returns


In [6]:
# Page 20 in full - it is short, and the unit line matters.
print(reader.pages[19].extract_text())

MINISTRY OF FINANCE 
 
20 
 
Table 2.4 Top-ups to Endowment and Trust Funds in FY2024 
 
 Estimated FY2024 
($ million) 
Goods and Services Tax Voucher Fund 6,000 
Future Energy Fund 5,000 
Edusave Endowment Fund  2,000 
Financial Sector Development Fund 2,000 
National Productivity Fund 2,000 
National Research Fund 1,800 
Progressive Wage Credit Scheme Fund 1,000 
Skills Development Fund 500 
Public Transport Fund 50 
Legal Aid Fund 2 
Total 20,352 
 
 
              


Two candidates, and they are not the same quantity:

| Page | Value | Unit | Year |
|---|---|---|---|
| 8 | 24.32 | $billion | Revised FY2023 |
| **20** | **20,352** | **$million** | **Estimated FY2024** |

Different year *and* different unit. Taking the wrong one is not a rounding
error - 24.32 and 20,352 differ by three orders of magnitude before the year is
even considered.

**Decision: 20,352 million from page 20**, as cited.

Note this field breaks the pattern of the others: it is the FY2024 figure, while
fields 1, 2 and 5 are FY2023. That follows from the page citations rather than
from any consistent year choice, and the prompt has to allow for it.

## Field 4: Taxes named in Operating Revenue

Requirements cite pages 5-6. The trap here is not ambiguity but truncation - the
first sentence reads like a complete list and is not.

In [7]:
text = reader.pages[4].extract_text() or ""
summary = re.search(r"This increase is mainly due to[^.]+\.", text)
print("The summary sentence on p.5:\n")
print(" ", " ".join(summary.group(0).split()))

named_in_summary = re.findall(
    r"(?:Corporate Income|Personal Income|Goods and Services|Other|Assets|Betting)\s+Tax(?:es)?"
    r"|Vehicle Quota Premiums",
    summary.group(0),
)
print(f"\n  -> {len(named_in_summary)} taxes named there")

The summary sentence on p.5:

  This increase is mainly due to higher collections from Corporate Income Tax, Other Taxes, Vehicle Quota Premiums, Personal Income Tax, Assets Taxes, and Betting Taxes, partially offset by lower collections from the Goods and Services Tax.

  -> 7 taxes named there


In [8]:
# Every tax-like name across both cited pages.
PATTERN = (
    r"[A-Z][A-Za-z',\- ]*?(?:Tax|Taxes|Duty|Premiums|Levy|Contributions|Charges)"
)
found = []
for page_no in (5, 6):
    page_text = reader.pages[page_no - 1].extract_text() or ""
    for name in re.findall(PATTERN, page_text):
        cleaned = " ".join(name.split())
        if cleaned not in found and len(cleaned) > 6:
            found.append(cleaned)

print(f"{len(found)} distinct tax-like names on pages 5-6:\n")
for name in found:
    print("   ", name)

12 distinct tax-like names on pages 5-6:

    Corporate Income Tax
    Other Tax
    Vehicle Quota Premiums
    Personal Income Tax
    Assets Tax
    Betting Tax
    Goods and Services Tax
    Collections from Other Tax
    Foreign Worker Levy
    Water Conservation Tax
    Charge, and Annual Tonnage Tax
    Casino Tax


The summary sentence names 7. Reading both pages to the end finds roughly twice
that, because taxes are introduced one per paragraph after the summary.

**Decision: at least 12 names, with the main taxes required.** Scored on count
and required members rather than exact equality - the regex above is a rough
guide, and a defensible extraction may or may not include marginal entries like
"Others" or split "Fees and Charges". What matters is that the model did not
stop at the summary sentence.

## Field 5: Overall Fiscal Position

Requirements cite page 8.

In [9]:
find("OVERALL FISCAL POSITION|Overall Fiscal Position", context=120)

  p 3: Overall Fiscal Position for FY2023 5 1.2 Operating Revenue 5 1.3 Total Expenditure 6 1.4 Special Transfers 7 1.5 Net


  p 5: Overall Fiscal Position for FY2023 The basic deficit is revised to $5.4 billion (0.8% of GDP). After factoring in To


  p 8: OVERALL FISCAL POSITION 1.72 (0.35) (3.57) Note: Figures may not add up due to rounding. Negative figures are shown i
  p13: Overall Fiscal Position for FY2024 is a surplus of $0.8 billion (0.1% of GDP). The FY2024 Budget is summarised in Tab


  p16: OVERALL FISCAL POSITION (3.57) 0.78 Note: Figures may not add up due to rounding. Negative figures are shown in paren


  p24: Overall Fiscal Position for FY2018 to FY2024 ($ million) BLANK FY2018 FY2019 FY2020 FY2021 FY2022 FY2023 FY2024 BLA
  p24: Overall Fiscal Position 3,338 845 (51,567) 1,880 1,716 (3,571) 778
  p25: Overall Fiscal Position for FY2018 to FY2024 (% of GDP)1 BLANK FY2018 FY2019 FY2020 FY2021 FY2022 FY2023 FY20
  p25: Overall Fiscal Position 0.7% 0.2% (10.5%) 0.3% 0.3% (0.5%) 0.1%


In [10]:
# The column headers that give the row its meaning.
text = reader.pages[7].extract_text() or ""
for line in text.splitlines():
    if any(k in line for k in ("Actual", "Estimated", "Revised", "OVERALL FISCAL")):
        print(" ", line.strip()[:100])

  BLANK    Revised FY2023
  BLANK Actual Estimated Revised Compared to
  BLANK FY2022 FY2023 FY2023 Actual Estimated
  OVERALL FISCAL POSITION 1.72 (0.35) (3.57)


The row reads `OVERALL FISCAL POSITION 1.72 (0.35) (3.57)`, and the header line
gives the columns: Actual FY2022, Estimated FY2023, Revised FY2023.

| Column | Value | Meaning |
|---|---|---|
| Actual FY2022 | 1.72 | a surplus, and the only realised outturn |
| Estimated FY2023 | (0.35) | forecast at Budget 2023 |
| **Revised FY2023** | **(3.57)** | **revised - the target** |

Two traps in one row. **Position**: the first number is not the answer, and
nothing but the header says so. **Parentheses**: (3.57) means -3.57, so a model
that ignores the convention returns a surplus where the document reports a
deficit - right magnitude, wrong sign.

Page 5 also states this in prose as "a deficit of $3.6 billion", which is the
same figure less precisely. The citation says page 8, so 3.57 is the answer -
the opposite of the choice made for Corporate Income Tax, and for the same
reason: follow the cited page.

**Decision: -3.57 from page 8.**

## Confirmed values

These are written to `evaluation/expected.yaml`. The cell below checks the file
matches what was derived above, so the two cannot drift apart.

In [11]:
from src.evaluation import load_expected

expected = load_expected()

DERIVED = {
    "corporate_income_tax": (28.4, 5),
    "corporate_income_tax_yoy": (17.0, 5),
    "total_top_ups": (20352.0, 20),
    "fiscal_position": (-3.57, 8),
}

for field, (value, page) in DERIVED.items():
    stored = expected[field]
    agrees = stored["value"] == value and stored["page"] == page
    print(
        f"{field:28s} {value:>10.2f} p{page:<3d} "
        f"{'matches expected.yaml' if agrees else 'MISMATCH'}"
    )

taxes = expected["operating_revenue_taxes"]
print(
    f"\n{'operating_revenue_taxes':28s} at least {taxes['min_count']} names, "
    f"{len(taxes['must_include'])} required"
)

corporate_income_tax              28.40 p5   matches expected.yaml
corporate_income_tax_yoy          17.00 p5   matches expected.yaml
total_top_ups                  20352.00 p20  matches expected.yaml
fiscal_position                   -3.57 p8   matches expected.yaml

operating_revenue_taxes      at least 12 names, 8 required


## What this exploration establishes

**Every field has plausible wrong answers in the document.** Not near-misses -
figures that are correct for a different year, a different unit, or a different
quantity. Corporate Income Tax alone has seven.

**The page citation is what disambiguates.** Nothing in a number says which year
or unit it belongs to. Restricting the model to the cited page eliminates the
alternatives before it reads anything, which is why `config.yml` binds each
field to a page and the prompt enforces it.

**Consistency was not available.** Fields 1, 2 and 5 are FY2023; field 3 is
FY2024. That follows from the citations, not from a choice - so any assumption
of "one target year" would be wrong for at least one field.

**Two conventions must be handled or the answer is silently wrong:**
parenthesised negatives (a sign error), and the $million/$billion split (a
1000x error).

**Scoring must check the page, not just the value.** 28.4 read from page 8 would
be the right number obtained by ignoring the instruction - it would not survive
a change to the document, so `src/evaluation.py` fails it.